# 00 · Getting the data

**Read this first.** This notebook explains the data for our Sprint 2 project, gets it onto your
machine, and shows how to load it. Run it top to bottom once; after that, any notebook can load the
data with two lines (see section 4).

The question we're answering: *does the text of a company's annual report (10-K) predict how volatile
its share price will be over the next year, beyond what last year's volatility already tells us?*
The full plan is in [`Oscar/PLAN.md`](../Oscar/PLAN.md).

## 1. What the data is

**27,142 annual reports** from US public companies, filed 1996–2006.

| Year | 1996 | 1997 | 1998 | 1999 | 2000 | 2001 | 2002 | 2003 | 2004 | 2005 | 2006 |
|---|---|---|---|---|---|---|---|---|---|---|---|
| Reports | 1,258 | 1,976 | 2,142 | 2,220 | 2,151 | 2,299 | 2,587 | 3,217 | 3,168 | 3,121 | 3,003 |

**Where it comes from**

- **Labels and Item 7 text:** the 10-K Corpus (Kogan et al., 2009). For each report it gives the
  share-price volatility in the 12 months *before* filing and the 12 months *after*, computed from
  CRSP, the standard academic price database.
- **Item 1 text, and most of Item 7:** EDGAR-CORPUS (Loukas et al., 2021), which has every 10-K
  already split into sections. Matched to Kogan's reports by SEC company number (CIK) and year.

**The two sections we use**

- **Item 1, the business description:** what the company does, its markets and competitors.
- **Item 7, management's discussion (MD&A):** management's account of the year. Taken from
  EDGAR-CORPUS, except where EDGAR's copy is under 100 words (usually a "see the annual report"
  stub), when Kogan's copy is used instead. Column `item7_source` says which one
  (worked out in `Oscar/03-Item7-Source.ipynb`).

**What has already been done to the text: FinBERT**

Reading 27,000 reports with FinBERT takes about 11 hours, so Oscar ran it once
(`Oscar/extract_finbert.py`) and shared the output. You never need to run FinBERT yourself.

1. Each section was cut into **chunks of 510 tokens** (roughly 400 words). FinBERT can't read more
   than 512 at once.
2. Each chunk went through **FinBERT** (`ProsusAI/finbert`), a BERT model further trained on
   financial text. We keep its summary of the chunk (the "CLS" vector): **768 numbers per chunk**.
3. At most **64 chunks** per section are kept. Only 0.3% (Item 1) and 0.6% (Item 7) of reports are
   longer than that. A typical section is 14–16 chunks.
4. Numbers are stored at half precision (float16) to halve the file size. A check against full
   precision showed no meaningful difference.

So each report becomes **a short sequence of 768-number vectors per section**, plus its two
volatilities. Our attention network takes the sequence; simpler models can average it into one vector.

**The labels**

- `logvol_before`: log of volatility over the 12 months before filing. **The baseline input.**
- `logvol_after`: log of volatility over the 12 months after filing. **What we predict.**

"Log volatility" means the natural log of the standard deviation of daily returns. That's why the
values are negative (around −3.4 typically). We score models by mean squared error on this,
as the 2009 paper does.

## 2. Get the files

**Step 1: add the shared OneDrive folder.** Oscar has shared a OneDrive folder called **`finbert`**
with you (check your Bristol email or OneDrive's *Shared* page). Open it in OneDrive on the web and
click **Add shortcut to My files**. It then syncs to your computer like your other OneDrive files.
It's about 1.5 GB. If OneDrive only keeps files "online", right-click the folder and choose
**Always keep on this device** so it doesn't stall mid-load.

It contains:

| File | What it is |
|---|---|
| `1996_000.npz` … `2006_012.npz` | 114 files of FinBERT vectors, 250 reports each |
| `manifest.json` | exact settings of the FinBERT run (model version, chunk size, …) |
| `kogan_edgar_merged.csv` | the labels and company details for every report |

**Step 2: set up Python**, if you haven't already. From the repo root:
`python3 -m venv .venv`, `source .venv/bin/activate`, `pip install -r requirements.txt`.
This notebook only needs numpy, pandas and matplotlib.

**Step 3: run the cells below.** They find the folder, then join everything into three files in
`data/processed/` of your copy of the repo.

**Never commit these data files.** `data/processed/` is in `.gitignore` already; GitHub would reject
files this size anyway.

### Find the shared folder

The next cell looks in the usual OneDrive locations. If it can't find the folder, paste the folder's
path into `FINBERT_DIR` (on a Mac: right-click the folder in Finder, hold Option, *Copy as Pathname*).

In [ ]:
import shutil
import sys
from pathlib import Path

FINBERT_DIR = None   # e.g. Path("/Users/you/OneDrive - University of Bristol/finbert") if not found

# The repo root is the folder above this notebook's folder.
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "Oscar"))   # so we can use Oscar/build_dataset.py

def find_finbert():
    home = Path.home()
    onedrives = [home / "OneDrive - University of Bristol",
                 home / "Library" / "CloudStorage" / "OneDrive-UniversityofBristol"]
    for od in onedrives:
        for candidate in [od / "finbert", od / "DST" / "finbert"]:
            if (candidate / "manifest.json").exists():
                return candidate
    return None

src = Path(FINBERT_DIR) if FINBERT_DIR else find_finbert()
assert src is not None and (src / "manifest.json").exists(), "Folder not found: set FINBERT_DIR above."
n_files = len(list(src.glob("*_[0-9][0-9][0-9].npz")))
print(f"Found the shared folder with {n_files} FinBERT files (expect 114).")

### Build the dataset

This copies the labels into `data/processed/`, then stacks the 114 files into one array per section
and lines them up with the labels. It takes about a minute (longer if OneDrive is still downloading).
You only need to do it once, or again if Oscar announces a new FinBERT run.

In [ ]:
from build_dataset import build, load_dataset, chunks

processed = ROOT / "data" / "processed"
processed.mkdir(parents=True, exist_ok=True)
shutil.copy(src / "kogan_edgar_merged.csv", processed / "kogan_edgar_merged.csv")

build(src)

This made three files in `data/processed/`:

| File | Contents |
|---|---|
| `dataset.csv` | one row per report (details below) |
| `item1_emb.npy` | every Item 1 chunk vector stacked into one table: 444,394 rows × 768 |
| `item7_emb.npy` | the same for Item 7: 498,607 rows × 768 |

Columns of `dataset.csv`:

| Column | Meaning |
|---|---|
| `key` | Kogan's report ID |
| `year`, `date` | year and date of filing |
| `cik`, `name` | SEC company number and company name |
| `logvol_before` | log volatility, 12 months before filing (**baseline input**) |
| `logvol_after` | log volatility, 12 months after filing (**target**) |
| `item7_source` | `edgar` or `kogan`: where the Item 7 text came from |
| `item1_n`, `item7_n` | number of chunks kept for this report (at most 64) |
| `item1_true`, `item7_true` | number of chunks before cutting at 64 |
| `item1_start`, `item7_start` | the row where this report's chunks start in the `.npy` file |

Why the `start` columns: reports have different numbers of chunks, so they can't sit in one
rectangular array. Instead all chunks are stacked, and report *i*'s Item 7 chunks are rows
`item7_start[i]` to `item7_start[i] + item7_n[i]`. The `chunks()` helper does this for you.

## 3. Check it worked

In [ ]:
df, emb = load_dataset()
print(f"{len(df):,} reports, years {df.year.min()}-{df.year.max()}")
print(f"Item 1 chunks: {emb['item1'].shape},  Item 7 chunks: {emb['item7'].shape}")
assert len(df) == 27142 and emb["item7"].shape == (498607, 768), "Something is missing: ask Oscar."
df.head()

## 4. How to use it in your own notebooks

```python
import sys; sys.path.insert(0, "../Oscar")
from build_dataset import load_dataset, chunks
df, emb = load_dataset()
```

`load_dataset()` is quick: the big arrays are *memory-mapped*, meaning they stay on disk and only
the rows you use are read.

**One report as a sequence of chunks.** This is the input for the attention network:

In [ ]:
i = 0
seq = chunks(df, emb, i, "item7")
print(df.loc[i, "name"], df.loc[i, "year"], "| Item 7 as", seq.shape, "(chunks x 768)")

**Every report as one vector.** Averaging each report's chunks gives a plain
(reports × 768) table that scikit-learn or a simple MLP can use directly.
`np.add.reduceat` adds up each report's block of rows in one go.

In [ ]:
import numpy as np

def mean_pool(section):
    sums = np.add.reduceat(emb[section], df[f"{section}_start"].to_numpy(), axis=0, dtype=np.float32)
    return sums / df[f"{section}_n"].to_numpy()[:, None]

X7 = mean_pool("item7")
print(X7.shape)

**Splitting by year.** We always train on earlier years and test on a later one, never mixing them,
so a model never sees the future. For example, testing on 2006 with the five years before it, as the 2009
paper did. The simplest baseline is to predict next year's volatility as last year's:

In [ ]:
train = df.year.between(2001, 2005).to_numpy()
test = (df.year == 2006).to_numpy()

mse = np.mean((df.logvol_after[test] - df.logvol_before[test]) ** 2)
print(f"train {train.sum():,} reports, test {test.sum():,} reports")
print(f"Baseline 'same as last year' MSE on 2006: {mse:.4f}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(df.logvol_before, df.logvol_after, s=2, alpha=0.2)
lims = [-8, 0]
ax.plot(lims, lims, color="k", lw=1)
ax.set(xlim=lims, ylim=lims, xlabel="log volatility, year before filing",
       ylabel="log volatility, year after filing", title="Last year's volatility is a strong baseline")
plt.show()

## 5. Things to know

- **One odd label.** The lowest `logvol_after` is −7.92, far below the rest (the middle half lies
  between −3.9 and −3.0). Worth deciding as a group whether to keep it.
- **Chunks are cut at 64.** Only a handful of reports are affected; `*_true` shows the full length.
- **Item 7 source.** About 2% of reports use Kogan's copy of Item 7 instead of EDGAR-CORPUS's.
  We might check that results don't depend on this.
- **Years 1996–2006 only.** There's no Risk Factors section (Item 1A) for most of these years.
  See the limitations in `PLAN.md`.
- **Don't edit the files in the shared folder.** If something looks wrong, tell Oscar; it's
  everyone's copy.

### References

- Kogan, Levin, Routledge, Sagi & Smith (2009). *Predicting Risk from Financial Reports with
  Regression*. NAACL. https://aclanthology.org/N09-1031/
- Loukas et al. (2021). *EDGAR-CORPUS: Billions of Tokens Make The World Go Round*.
  https://aclanthology.org/2021.econlp-1.2
- Araci (2019). *FinBERT: Financial Sentiment Analysis with Pre-trained Language Models*.
  https://arxiv.org/abs/1908.10063